## Example: Run COLMAP Pipeline

Now you can run COLMAP commands in the container across different cells.

# Running COLMAP on a Set of Images
----
In this notebook, we'll walk through how to run COLMAP on your own set of images. Reconstructing a Gaussian Splat radiance field requires camera location and orientations and a sparse point cloud of initialization, along with the input images. COLMAP is one way to get the camera information and a sparse point cloud.
The notebook walks through a common way to run COLMAP on outdoor aerial imagery.

## Gather Data

Your reconstruction can only be as good as the data you start with, because of this it's crucial to have good images. Make sure you have good lighting and high overlap between frames. COLMAP needs to be able to match objects between frames. For outdoor scenes there are some best practices you can follow to get good results.

If you have one main object or area you're focusing on it is recommended to circle or orbit it with a camera equipped aerial vehicle. For larger outdoor scenes with no one specific focus, either overlapping circular orbits or traditional oblique mapping flight lines produce good quality results. 
<table style="width:80%;">
  <tr>
    <td style="width:33%; text-align:center;">
      <img src="images/safety_park_camera_poses.png" alt="Orbit Camera Positions" style="width:100%;">
      <div><strong>Orbit:</strong> Camera positions for a scene with a single area of focus. Cameras point towards the center of the scene wile orbiting around it. This scene is an NVIDIA collected scene known as Safety Park. </div>
    </td>
    <td style="width:33%; text-align:center;">
      <video src="images/Flight.mp4" style="width:100%;" controls></video>
      <div><strong>Oblique Camera:</strong> You can either have 5 different cameras or a smart camera to capture each of the 5 views. A ariel vehicle equipped with the camera follows the above path for a large area data capture. Arrows designate which images from the oblique to keep. Diagram adapted from <a href="https://https://enterprise-insights.dji.com/blog/smart-oblique-capture">Enterprise</a> </div>
    </td>
  </tr>
  
</table>
Be warery of motion blur if you're flying too fast the frames might have too much blur to reconstruction a high feildlety scene.

Indoor scenes
<table style="width:80%;">
  <tr>
    <td style="width:33%; text-align:center;">
      <img src="images/nerfbox.jpg" alt="Semisphere Camera Positions" style="width:100%;">
      <div><strong>Semisphere:</strong> Camera positions for a scene with a single object or small area of focus. Cameras point towards the object in placement that resembles a semisphere around the object. Diagram from <a href="https://github.com/NVlabs/instant-ngp/blob/master/docs/nerf_dataset_tips.md">nerf_dataset_tips.md</a>.</div>
    </td>
    <td style="width:33%; text-align:center;">
      <video src="images/room_video.mp4" style="width:100%;" controls></video>
      <div><strong>Indoor Scenes/rooms:</strong> If you'd like to reconstruct an interior, such as a whole room its important to move the camera location. COLMAP tends to perform poorly if the camera location doesn't move, so avoid standing still and just rotating the camera. Above is an example of camera positions in the room dataset from the <a href="https://arxiv.org/abs/2111.12077">Mip-NeRF 360</a> collection. </div>
    </td>

## From Video to Frames

Usually data is collected in the form a video, at 30 or 60 fps a short video of our scene can contain thousands of images. For a scene with just one focus, like safety park or the room, we only need between 100 - 200 images to get a good reconstruction.

We can rip frames for a video easily using [FFmpeg](https://www.ffmpeg.org/), a command line tool for processing multimedia content.

FFmpeg is available on Linux, Windows and macOS. Go to the [download](https://www.ffmpeg.org/download.html) page and follow the download and installation instructions for your OS.

We can use it to grab 1 frame per second from our input video. In this notebook we will using a video of Safety Park as our input scene.

First lets create a folder where will store our data. Lets call it safety_park. This folder will store our original input image and the output of our COLMAP run. fVDB Realty cCapture expects a the file structure will be creating, so make sure you're following similar steps if using your own data. Just change the folder and video name from safety_park to your data set name.

In [ ]:
!mkdir data/safety_park

Now run FFmpeg to save out frames.

In [ ]:
!ffmpeg -i safety_park.mp4 -vf "fps=1" data/safety_park/images_raw/frame_%04d.png

You can experiment with changing how many frames per second you save a frames. Aim to get a least 100 images from your data set. 

## Downloading COLMAP


[COLMAP](https://colmap.github.io/) is a SfM (structure-from-motion) and MVS (multi-view stereo) pipeline. 

COLMAP has prebuilt binaries, Docker containers, or can be built from source. If using Linux we recommend you build for source or use a Docker container for CUDA support.

Got to COLMAP's [installation page](https://colmap.github.io/install.html#installation) and follow the instructions for your preferred method of installation.

Once you've installed COLMAP make sure it works by running it with the help command below.


In [ ]:
!colmap -h

We need 2 important things from COLMAP, and estimate of the camera locations and poses and sparse point cloud of the scene. Several COLMAP commands need to run to get our desired output, for ease we have gathered them in 1 easy to use python script.

1) Feature Detection & Extraction
The first step in the COLMAP pipeline is to find and describe SIFT features. Features are small, distinctive patches in an image (like corners or textured regions) that are likely to be recognizable in other images. COLMAP detects these points and computes a numerical descriptor for each one so they can be compared across images.

2) Feature Matching
With a collection of features, the next step is to match them between images. For a pair images, with a feature from one, COLMAP will search for the closest match in the other. Some matches will be poor, to weed these out COLMAP run geometric checks to make sure the matches make sense.

3) Mapping
In this step, COLMAP reconstructs a sparse 3D point cloud and estimates the camera poses. It starts with a pair of images that share many good matches and triangulates 3D points from their correspondences. Then it adds images one by one: for each new image, it estimates the camera pose and triangulates additional points from overlapping views. This yields both an estimate of the 3D structure of the scene and the position and orientation of each camera. During this process, COLMAP periodically runs bundle adjustment, a non-linear optimization that jointly refines camera parameters and 3D point locations to minimize reprojection error and improve the final reconstruction

4) Model Alignment
If we have georeferenced images, like we do with Safety Park, we can align and scale the model to match real-world GPS coordinates. This produces a reconstruction in an Earth-referenced coordinate system. If we do not have GPS info, we can use COLMAP’s principal plane / plane alignment, which attempts to orient the model so that it is “right side up” and more convenient to work with.

5) Image Undistortion
To help radiance field methods like NeRF and 3D Gaussian Splatting we will undisort our images. This means we create a new set of images that resemble the original if we had instead taken the images with a an ideal pinhole camera, as most radiance field methods expect this ideal camera format. The corresponding camera parameters and sparse model are adjusted to match these undistorted images.

Below is an example command for Safety Park. Safety Park does have GPS info so we will run with geoalign. If a GPU is available it will be automatically used. 

We recommend running the command below in a terminal, not in the notebook, as the notebook suppresses the logging info until the end.

In [ ]:
!python run.py -s data/safety_park2/ %run run.py -s data/safety_park --global_ba --geoalign

What parameters you'll need will vary on your data set. Let's go over a few.


Before we train a 3D Gaussian Splat lets make sure our COLMAP run produced good results. To do this we will view teh sparse reconstruction and camera positions in the viewer.

In [ ]:
!colmap gui

To open the scene, navigate to **File > Import model**. Once the file explorer pops up select the *sparse/0* folder within your data folder. For Safety Park we are navigating to *data\safety_park\sparse\0*.

Important: Use **Import model** not **Import Model from**

You can change the sparse points' size by scrolling while holding **CTRL** and the camera symbol sized by scrolling while holding **ALT**. 

Now that the scene is loaded take a close look at it. Does your scene look right? Can you recognize locations and objects from you original imagery? Do the camera locations make sense? If you answered yes to all these questions you're ready to train a Gaussian Splat!

Some things to look out for that are indicative of a poor result
- Areas of the scene that should reside next to each other are split far apart in the reconstruction
- Camera or cluster of point far away from the main scene
- The points look random, you can't make out specific objects or areas 
- The camera positions are all over the place. If your frames came from a video, your camera positions should follow a traceable path.

Don't worry if you're result aren't what you wanted. There a few thing we can do to improve.
The most common cause of a bad COLMAP spaarse reconatction is bad data. Take a look back at your data. 

Do your frames have 60 - 80% overlap? Are objects moving in the scene


Yuu can change the sparse points' size by scrolling while holding **CTRL** and the camera symbol sized by scrolling while holding **ALT**. 